In [22]:
import pandas as pd
import re

In [23]:
original_df = pd.read_csv("../data/nyc_2024_data_selected.csv")
print(f"The data frame has {original_df.shape[0]} rows and {original_df.shape[1]} columns.")
# print(original_df.head())

The data frame has 30008 rows and 62 columns.


In [24]:
print(original_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 30008 entries, 0 to 30007
Data columns (total 62 columns):
 #   Column                                                           Non-Null Count  Dtype  
---  ------                                                           --------------  -----  
 0   property_id                                                      30008 non-null  int64  
 1   primary_property_type_self                                       30008 non-null  str    
 2   national_median_reference                                        29479 non-null  str    
 3   list_of_all_property_use                                         30008 non-null  str    
 4   largest_use_gfa_sqft                                             30001 non-null  float64
 5   _2nd_largest_property_use                                        8313 non-null   str    
 6   second_use_gfa_sqft                                              8313 non-null   float64
 7   _3rd_largest_property_use                          

***First, we drop the `property_id` column as it is not useful for a ML algorithm which is just used as an identifier.***

In [25]:
df = original_df.drop(columns=["property_id"])
# print(df.head())

In [26]:
print(df["metered_areas_water"].describe())
print("--------------------------------------------------")
print(df["electric_distribution_utility"].describe())

count              21820
unique                 5
top       Whole Property
freq               21795
Name: metered_areas_water, dtype: object
--------------------------------------------------
count                             29991
unique                                8
top       Consolidated Edison Co-NY Inc
freq                              29934
Name: electric_distribution_utility, dtype: object


***we drop the `national_median_reference` as it is just the nation-median category which overlaps the `primary_property_type_self`, drop the `list_of_all_property_use` is a very long string which is also overlaps with the other related columns such as `primary_property_type_self` and some gross columns, drop the `metered_areas_water` and `electric_distribution_utility` while most them are the same campany as listed above which is not useful for the training. Moreover, the `number_of_activate_energy_meters_2`, `number_of_active_it_meters`, and `aggregate_meter_s_electric_number_of_individual_meters_included` all overlap with the `number_of_active_energy_meters` whici is the total number of the meters. Plus, we also drop `_2nd_largest_property_use`, `second_use_gfa_sqft`, `_3rd_largest_property_use`, and `third_use_gfa_sqft` as they are considered redundant with the property-level sqft and have too much missing values.***

In [27]:
cols_to_drop = [
    "national_median_reference",
    "list_of_all_property_use",
    "metered_areas_water",
    "electric_distribution_utility",
    "number_of_active_energy_meters_2",
    "number_of_active_it_meters",
    "aggregate_meter_s_electric_number_of_individual_meters_included",
    "_2nd_largest_property_use",
    "second_use_gfa_sqft",
    "_3rd_largest_property_use",
    "third_use_gfa_sqft",
]

df = df.drop(columns=cols_to_drop)
print(df.shape)

(30008, 50)


***Here, as the target variable is the Site EUI which is the density measurement. To avoid overlapping, we drop the columns which means the total number of a property, just keep the density columns which is the same measurement from a different perspective.*** 

In [28]:
cols_to_drop = [
    "multifamily_housing_number",
    "multifamily_housing_total",
    "multifamily_housing_number_4",
    "multifamily_housing_number_6",
    "office_number_of_workers",
]

df = df.drop(columns=cols_to_drop)
print(df.shape)

(30008, 45)


***Since some of the data fields in the later sections of the dataset are specific to certain groups, we need to identify the records associated with different Property IDs within each group. Then, for each group, we check whether there are any missing values in the feature descriptions that are applicable to that particular group.***

In [29]:
property_types = {
    "Medical Office": "medical_office_",
    "K-12 School": "k_12_school_",
    "Multifamily Housing": "multifamily_housing_",
    "Office": "office_",
    "Restaurant": "restaurant_",
    "Parking": "parking_",
    "Retail Store": "retail_store_",
}

uses = original_df.loc[df.index, "list_of_all_property_use"]

property_groups = {}

for property_type in property_types:
    pattern = rf"(?:^|,\s*){re.escape(property_type)}\s*\("

    identifier = uses.str.contains(pattern, regex=True, na=False)

    property_groups[property_type] = df.loc[identifier].copy()

    print(f"{property_type} has {len(property_groups[property_type])} records.")

Medical Office has 938 records.
K-12 School has 1946 records.
Multifamily Housing has 20320 records.
Office has 3558 records.
Restaurant has 1090 records.
Parking has 3092 records.
Retail Store has 2495 records.


In [30]:
results = []

for property_type, group_df in property_groups.items():
    prefix = property_types[property_type]

    for col in group_df.columns:
        if col.startswith(prefix):
            results.append({
                "property_use": property_type,
                "feature": col,
                "num_of_records": len(group_df),
                "num_of_unique": group_df[col].nunique(),
                "missing_values_percent": group_df[col].isnull().mean() * 100,
            })

summary = pd.DataFrame(results)

display(summary.sort_values(["property_use", "missing_values_percent"], ascending=[True, False]).round(2))

,property_use,feature,num_of_records,num_of_unique,missing_values_percent
7,K-12 School,k_12_school_gross_floor_area,1946,1412,0.10
0,Medical Office,medical_office_gross_floor,938,745,0.00
1,Medical Office,medical_office_number_of,938,459,0.00
2,Medical Office,medical_office_number_of_1,938,8,0.00
3,Medical Office,medical_office_number_of_2,938,383,0.00
4,Medical Office,medical_office_percent_that,938,13,0.00
5,Medical Office,medical_office_percent_that_1,938,12,0.00
6,Medical Office,medical_office_weekly,938,82,0.00
18,Multifamily Housing,multifamily_housing_resident,20320,5,42.54
8,Multifamily Housing,multifamily_housing_government,20320,2,40.90


***Deleting the missing values records, as we not want any missing values. If a record belongs to a particular group but its building features do not match the feature values associated with that group, we remove that record.***

In [31]:
df_clean = df.copy()

for property_type, group_df in property_groups.items():
    prefix = property_types[property_type]

    # find a corresponding column in this group
    group_cols = [
        col for col in group_df.columns
        if col.startswith(prefix)
    ]

    # find any missing value in this record
    missing_rows = group_df[group_df[group_cols].isnull().any(axis=1)]

    # drop it, the error="ignore" is to avoid KeyError if the index is not found in df_clean as there maybe some records belongs to multiple property_use
    df_clean = df_clean.drop(index=missing_rows.index, errors="ignore")

# drop if the target variable is missing
df_clean = df_clean.dropna(subset=["site_eui_kbtu_ft"])

print(df_clean.shape)

(19742, 45)


In [32]:
print(df_clean.columns.tolist())

['primary_property_type_self', 'largest_use_gfa_sqft', 'year_built', 'number_of_buildings', 'occupancy', 'site_eui_kbtu_ft', 'medical_office_gross_floor', 'medical_office_number_of', 'medical_office_number_of_1', 'medical_office_number_of_2', 'medical_office_percent_that', 'medical_office_percent_that_1', 'medical_office_weekly', 'k_12_school_gross_floor_area', 'multifamily_housing_government', 'multifamily_housing_gross', 'multifamily_housing_number_1', 'multifamily_housing_number_2', 'multifamily_housing_number_3', 'multifamily_housing_percent', 'multifamily_housing_percent_1', 'multifamily_housing_total_1', 'multifamily_housing_number_5', 'multifamily_housing_number_7', 'multifamily_housing_resident', 'office_gross_floor_area_ft', 'office_number_of_computers', 'office_percent_that_can_be', 'office_percent_that_can_be_1', 'office_weekly_operating_hours', 'office_worker_density_number', 'parking_open_parking_lot', 'parking_completely_enclosed', 'parking_partially_enclosed', 'restauran

In [33]:
print(df_clean["year_built"].max())

2025


In [34]:
print(df_clean.loc[df_clean["year_built"] == 2025])

      primary_property_type_self  largest_use_gfa_sqft  year_built  \
29516                     Office               47031.0        2025   

       number_of_buildings  occupancy  site_eui_kbtu_ft  \
29516                    1          5               0.0   

       medical_office_gross_floor  medical_office_number_of  \
29516                         NaN                       NaN   

       medical_office_number_of_1  medical_office_number_of_2  ...  \
29516                         NaN                         NaN  ...   

       restaurant_weekly_operating  restaurant_worker_density  \
29516                          NaN                        NaN   

       retail_store_gross_floor  retail_store_number_of_walk  \
29516                       NaN                          NaN   

      retail_store_number_of_open  property_gfa_self_reported  \
29516                         NaN                     58987.0   

       number_of_active_energy_meters   borough   latitude  longitude  
29516    

***Drop this column, as this record's `year_built` does not belong to 2024 and before.***

In [35]:
df_clean.drop(df_clean[df_clean["year_built"] == 2025].index, inplace=True)
print(df_clean.shape)

(19741, 45)


***Although there are some zero values in the `number_of_buildings` column, these zeros should not be treated as missing values because they have actual meaning. As shown in the displayed table, these records still contain valid values for our target variable and provide useful information. Therefore, we should retain these records rather than remove them.***

In [36]:
# print(df_clean["number_of_buildings"].describe())
# print(df_clean["number_of_buildings"].value_counts().sort_index().to_string())
print(df_clean["number_of_buildings"].isnull().sum())
display(df_clean.loc[df_clean["number_of_buildings"] == 0, ["primary_property_type_self", "largest_use_gfa_sqft", "property_gfa_self_reported", "number_of_buildings", "site_eui_kbtu_ft"]])

0


,primary_property_type_self,largest_use_gfa_sqft,property_gfa_self_reported,number_of_buildings,site_eui_kbtu_ft
868,K-12 School,24210.0,24210.0,0,44.3
936,K-12 School,38500.0,38500.0,0,167.4
1036,K-12 School,49510.0,49510.0,0,33.9
1037,K-12 School,51830.0,51830.0,0,52.0
1059,K-12 School,74400.0,74400.0,0,61.9
...,...,...,...,...,...
29549,Pre-school/Daycare,4500.0,4500.0,0,71.6
29550,Pre-school/Daycare,15000.0,15000.0,0,7.2
29551,College/University,74189.0,74189.0,0,96.0
29552,Other - Lodging/Residential,16000.0,16000.0,0,165.1


***Since our goal is to calculate the EUI for the entire site, we will retain `property_gfa_self_reported`. In contrast, `largest_use_gfa_sqft` only represents a portion of the site and therefore may not accurately reflect the characteristics of our target variable. As a result, we will drop the `largest_use_gfa_sqft` column.***

In [37]:
df_clean.drop(columns=["largest_use_gfa_sqft"], inplace=True)
print(df_clean.shape)

(19741, 44)


***We also noticed that many properties have a reported area of only 1 square foot, which is clearly unrealistic. These values can be considered outliers, as they may lead to significant deviations in our final predictions. Therefore, we removed these records to prevent unrealistic data from negatively affecting our model's performance.***

In [38]:
# print(df_clean["property_gfa_self_reported"].isnull().sum())
# print(df_clean.loc[df_clean["property_gfa_self_reported"] == 0])

In [39]:
# print(df_clean.loc[df_clean["property_gfa_self_reported"] == 1])
df_clean.drop(df_clean.loc[df_clean["property_gfa_self_reported"] == 1].index, inplace=True)
print(df_clean.shape)

(19737, 44)


***We also found that many records contain missing values for geographic information, including the regions where the properties are located and their latitude and longitude coordinates. Therefore, we removed all records with missing values in these demographic features.***

In [40]:
print(df_clean["borough"].isnull().sum())
print(df_clean["longitude"].isnull().sum())
print(df_clean["latitude"].isnull().sum())

683
686
686


In [41]:
df_clean.dropna(subset=["borough", "longitude", "latitude"], inplace=True)
print(df_clean.shape)

(19051, 44)


In [42]:
df_clean.to_csv("../data/nyc_2024_data_cleaned.csv", index=False)